# Kiểm tra nhúng và bộ đệm
Chạy thủ công khi đã cấu hình OpenRouter và dựng kho bằng `python -m app.kb.build`.
Sổ tay chỉ dùng truy vấn theo mẫu; không đưa văn bản ca bệnh vào nhúng.
Cấu hình OpenRouter dùng bộ đệm do `build_retriever` lắp; không hiển thị khoá.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "app").is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from app.container import build_container
from app.retrieval.query_builder import QueryBuilder
from app.retrieval.setup import build_retriever

container = build_container()
retriever = build_retriever(container)
builder = QueryBuilder(container.settings.retrieval)

In [ ]:
import numpy as np

from app.providers.cached_embedder import CachedEmbedder

embedder = retriever.embedder
if not isinstance(embedder, CachedEmbedder):
    raise ValueError("Chọn cấu hình nhúng OpenRouter có bộ đệm để chạy sổ tay này")

queries = [
    builder.for_finding("meropenem", "meropenem", "meropenem", 6, None),
    builder.for_finding("amoxicillin", "amoxicillin", "amoxicilin", 4, None),
]
sentences = [text for query in queries for text in query.van_ban]
if len(sentences) != 4:
    raise ValueError("Sổ tay cần cấu hình query_languages gồm vi và en")
for index, text in enumerate(sentences):
    print(index, text)
vectors = np.asarray(embedder.embed_documents(sentences))
norms = np.linalg.norm(vectors, axis=1)
print("Số chiều:", vectors.shape[1])
print("Chuẩn véc-tơ:", norms)
denominators = np.outer(norms, norms)
cosine = np.divide(vectors @ vectors.T, denominators,
                   out=np.zeros((len(sentences), len(sentences))),
                   where=denominators != 0)
print("Ma trận cosine:")
print(cosine)

In [ ]:
before = embedder.stats()
second = np.asarray(embedder.embed_documents(sentences))
np.testing.assert_allclose(second, vectors, atol=1e-6)
print("Bộ đệm trước lần hai:", before)
print("Bộ đệm sau lần hai:", embedder.stats())